In [ ]:
from pathlib import Path
import re
import rasterio
from rasterio.mask import mask
from rasterio.warp import transform_geom
from shapely.geometry import box, mapping

# =========================================================
# SETTINGS
# =========================================================
INPUT_DIR = Path(r"../West_Fl_Shelf/landsat_sst_outputs")
OUTPUT_DIR = INPUT_DIR.parent / "chopped_sst"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Small buffer in degrees
BUFFER_DEG = 0.05

# =========================================================
# AOI POINTS (lon, lat)
# west longitudes must be negative
# =========================================================
points = [
    (-83.475, 25.171),
    (-83.654, 25.704),
    (-83.086, 26.010),
]

# =========================================================
# BUILD A SQUARE POLY WITH SMALL BUFFER
# =========================================================
lons = [p[0] for p in points]
lats = [p[1] for p in points]

min_lon = min(lons)
max_lon = max(lons)
min_lat = min(lats)
max_lat = max(lats)

width = max_lon - min_lon
height = max_lat - min_lat
side = max(width, height) + 2 * BUFFER_DEG

center_lon = (min_lon + max_lon) / 2
center_lat = (min_lat + max_lat) / 2

half_side = side / 2

square_min_lon = center_lon - half_side
square_max_lon = center_lon + half_side
square_min_lat = center_lat - half_side
square_max_lat = center_lat + half_side

# shapely square in EPSG:4326
square_geom_4326 = box(square_min_lon, square_min_lat, square_max_lon, square_max_lat)

# POLY for your reference
POLY = [
    [square_min_lon, square_min_lat],
    [square_max_lon, square_min_lat],
    [square_max_lon, square_max_lat],
    [square_min_lon, square_max_lat],
    [square_min_lon, square_min_lat],
]

print("Generated square POLY (lon, lat):")
for p in POLY:
    print(p)

# =========================================================
# HELPER: extract first YYYYMMDD from filename
# =========================================================
def extract_first_date(name: str) -> str | None:
    m = re.search(r"(\d{8})", name)
    return m.group(1) if m else None

# =========================================================
# LOOP ALL TIFS AND CROP
# =========================================================
tif_files = sorted(INPUT_DIR.glob("*.tif"))

if not tif_files:
    print(f"No tif files found in: {INPUT_DIR}")

for tif_path in tif_files:
    try:
        out_date = extract_first_date(tif_path.name)
        if out_date is None:
            print(f"Skip (no date found): {tif_path.name}")
            continue

        out_path = OUTPUT_DIR / f"{out_date}.tif"

        with rasterio.open(tif_path) as src:
            if src.crs is None:
                print(f"Skip (no CRS): {tif_path.name}")
                continue

            # transform AOI from EPSG:4326 to raster CRS
            geom_src_crs = transform_geom(
                "EPSG:4326",
                src.crs,
                mapping(square_geom_4326)
            )

            cropped, cropped_transform = mask(
                src,
                [geom_src_crs],
                crop=True,
                nodata=src.nodata
            )

            out_meta = src.meta.copy()
            out_meta.update({
                "height": cropped.shape[1],
                "width": cropped.shape[2],
                "transform": cropped_transform
            })

            with rasterio.open(out_path, "w", **out_meta) as dst:
                dst.write(cropped)

        print(f"Saved: {out_path.name}")

    except Exception as e:
        print(f"Error processing {tif_path.name}: {e}")

print("Done.")